# chromatin_od re-ranking overhead — click-to-top-100 latency profile

Third in this series, alongside `tm_score_latency_profile.ipynb` and
`od_contrast_latency_profile.ipynb`. Same SETUP, same six PIPELINE stages, same config and
RNG seeding -- byte-identical code through stage 6. Question: **does ranking by
`chromatin_od` (production's name for the `od51` axis, per the accepted notebook's `AXES`
dict: `'chromatin_od': 'od51'`) cost as much as `od_contrast` did?**

**Why it should be cheaper, and by how much.** `od_contrast = od51 - od_ctx` needs two
`chromatin_density` passes: `od51` (window=51, 2,601 px) and `od_ctx` (window=121,
14,641 px). `chromatin_od` needs only `od51` -- so this measures a strict *subset* of
`od_contrast`'s stage 7, skipping the `od_ctx` loop entirely (which the second notebook
found was 79% of stage 7's cost) and the final subtraction.

A same-conversation back-of-envelope estimate (reusing `od_contrast`'s own measured
`t7a`+`t7b`+`t8`, which used a 60px pad sized for `od_ctx`) put the overhead at roughly
720ms mean. That estimate is **not accepted as-is here** -- it reused padding sized for a
window this notebook never touches, and it was never checked against a real oracle. This
notebook computes only what `chromatin_od` needs, with padding sized to `chromatin_od`'s
own requirement, and verifies the result against production's own `nan_rate` /
`largest_tie_block` for `arm='chromatin_od'` -- then checks the earlier estimate against
what actually ran, rather than assuming it held.

**Padding.** `OD_PAD_51 = 51 // 2 = 25` -- not the `od_contrast` notebook's `OD_PAD=60`
(sized for `od_ctx`'s 121px window, irrelevant here). A single-ROI smoke test (013.tiff, a
ROI whose candidates span the full width/height including pixel 0 and the last pixel)
verified two things before this notebook was built: (1) `od51` computed with a 25px pad
exactly matches `od51` computed with a 60px pad, elementwise, since `BORDER_REPLICATE`
padding beyond what a window actually reads changes nothing -- extra padding costs time,
not correctness; (2) `nan_rate`/`largest_tie_block` matched the production oracle exactly.

In [1]:
import gc
import os
import subprocess
import sys
import time

import cv2
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import chromatin as cm
from midog_utils import compare as cp
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

# ---------------------------------------------------------------------------------------
# Config -- identical to the first two notebooks / the accepted notebook's cell 1.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0

CHANNEL = 'hematoxylin_od'
METHOD = cv2.TM_CCOEFF
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 5.0
DEEP_FLOOR_Z = -1.5
MAX_PEAKS = 2_000_000

NMS_RADIUS_UM = ev.MIDOG_RADIUS_UM
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=1, flips=(False,), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
BORDER = CFG.patch_size // 2
OTSU_WINDOW = tm.BASE_SIZE

BUDGET = 100  # top-K for both tm_score and chromatin_od rankings

# chromatin_od-specific config. OD51_WINDOW/frac match the accepted notebook's od51 call
# exactly (frac defaults to chromatin.DEFAULT_FRAC=0.10). OD_PAD_51 is sized ONLY for this
# window -- not od_contrast notebook's OD_PAD=60, which was sized for od_ctx's 121px window.
OD51_WINDOW = 51
OD_PAD_51 = OD51_WINDOW // 2   # 25 -- exactly covers a 51px window, verified by smoke test

ORACLE_RAW_CSV = '../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv'
OD_CONTRAST_PER_ROI_CSV = 'od_contrast_latency_per_roi.csv'  # for the estimate-vs-actual check

print(f'BUDGET={BUDGET}, NMS radius = match radius = {NMS_RADIUS_UM} um, channel={CHANNEL}')
print(f'chromatin_od (od51): window={OD51_WINDOW} frac={cm.DEFAULT_FRAC} OD_PAD_51={OD_PAD_51}')

BUDGET=100, NMS radius = match radius = 7.5 um, channel=hematoxylin_od
chromatin_od (od51): window=51 frac=0.1 OD_PAD_51=25


## Helpers

Copied or reimplemented verbatim from the first notebook -- `suppress()` is inline in the accepted notebook, not part of any `midog_utils` module. `cpu_speed_limit()`/`wait_for_cool_cpu()` are the thermal-safety infrastructure added after the `od_contrast` notebook hit thermal throttling twice.

In [2]:
def draw_seed_with_retry(pool, rng, check_fn):
    """Draw a row via `rng.integers`; on failure drop it and redraw on the same stream."""
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, ref_xy):
    """NMS at `radius`, then drop the template's own self-correlation."""
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - ref_xy[0], c[:, 1] - ref_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def roi_files(images_dir=IMAGES_DIR):
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


def cpu_speed_limit():
    """Parse `CPU_Speed_Limit` from `pmset -g therm` -- 100 = full speed, lower = throttled."""
    try:
        out = subprocess.run(['pmset', '-g', 'therm'], capture_output=True, text=True,
                             timeout=5).stdout
        for line in out.splitlines():
            if 'CPU_Speed_Limit' in line:
                return int(line.strip().split('=')[-1].strip())
    except Exception:
        pass
    return None


def wait_for_cool_cpu(max_wait_s=300, poll_interval_s=5):
    """Block until `CPU_Speed_Limit` reads 100, or give up after `max_wait_s`."""
    waited = 0
    limit = cpu_speed_limit()
    while limit is not None and limit < 100 and waited < max_wait_s:
        time.sleep(poll_interval_s)
        waited += poll_interval_s
        limit = cpu_speed_limit()
    return limit, waited


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')
print(f'CPU_Speed_Limit at notebook start: {cpu_speed_limit()}')

14 ROIs on disk in ../images/extra_valid/
CPU_Speed_Limit at notebook start: 100


## Per-ROI timing function

Stages 1-6 are byte-identical to the first two notebooks. Stage 7 now has only two
sub-steps (pad, `od51` loop) -- no `od_ctx` loop, no subtraction, since `chromatin_od`
ranks directly on `od51`. Stage 8 ranks by `od51` instead of `od_contrast`.

In [3]:
def time_roi(fn, image_id, domain, anns):
    cpu_limit_start = cpu_speed_limit()
    gc.disable()

    # =================== SETUP (once per ROI; not part of click latency) ==============
    t0 = time.perf_counter()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    nms_radius = ev.radius_px(mpp, NMS_RADIUS_UM)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb

    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)

    rng = np.random.default_rng([SEED_INDEX, image_id])

    def _check(row):
        r = ss.tightened_template_box(gray_inv, float(row['cx']), float(row['cy']),
                                      otsu_window=OTSU_WINDOW)
        if r is None:
            return None
        if tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size) is None:
            return None
        return r

    seed, seed_tpl_probe, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    seed_ann_id = int(seed['ann_id'])
    click_cx, click_cy = float(seed['cx']), float(seed['cy'])
    t_setup = time.perf_counter() - t0

    # ======================= PIPELINE (click-to-top-100 latency) =======================
    stages = {}
    t_outer0_tm = time.perf_counter()

    # ---- Stage 1: refine the bounding box of the valid seed (single call, no retry) ---
    t1 = time.perf_counter()
    r = ss.tightened_template_box(gray_inv, click_cx, click_cy, otsu_window=OTSU_WINDOW)
    assert r is not None, f'{fn}: seed was pre-validated in SETUP but refused here'
    border_probe = tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size)
    assert border_probe is not None, f'{fn}: seed was pre-validated in SETUP but unreadable here'
    assert r == seed_tpl_probe, f'{fn}: single-shot refinement diverged from the SETUP search'
    base_size, tpl_cx, tpl_cy = r
    tpl_xy = (float(tpl_cx), float(tpl_cy))
    stages['t1_refine_seed_box_s'] = time.perf_counter() - t1

    # ---- Stage 2: patch + template build ----------------------------------------------
    t2 = time.perf_counter()
    patch = tm.read_padded_patch(hem, *tpl_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    stages['t2_patch_template_build_s'] = time.perf_counter() - t2

    # ---- Stage 3: template matching -----------------------------------------------------
    t3 = time.perf_counter()
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    stages['t3_template_matching_s'] = time.perf_counter() - t3

    # ---- Stage 4: threshold + peak extraction ------------------------------------------
    t4 = time.perf_counter()
    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    med, mad = tm.robust_stats(fused, valid)
    cut = med + DEEP_FLOOR_Z * mad
    centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, MAX_PEAKS)
    assert len(centers) < MAX_PEAKS, f'{fn}: MAX_PEAKS is binding, raise it'
    stages['t4_threshold_peak_extraction_s'] = time.perf_counter() - t4

    # ---- Stage 5: NMS + self-hit suppression -------------------------------------------
    t5 = time.perf_counter()
    c, s = suppress(centers, scores, nms_radius, tpl_xy)
    stages['t5_nms_selfhit_s'] = time.perf_counter() - t5
    assert bool(np.all(np.diff(s) <= 0)), f'{fn}: post-NMS pool is not score-descending'

    # ---- Stage 6: rank + top-100 by tm_score -------------------------------------------
    t6 = time.perf_counter()
    pool = pd.DataFrame({'cx': c[:, 0], 'cy': c[:, 1], 'score': s})
    top_tm = pool.sort_values('score', ascending=False, na_position='last',
                              kind='mergesort').head(BUDGET)
    stages['t6_rank_top100_s'] = time.perf_counter() - t6

    t_outer_tm_total = time.perf_counter() - t_outer0_tm

    # ---- Stage 7: chromatin_od (od51) feature on the same post-NMS pool ----------------
    # Only od51 -- not od31/od81/od_ctx/od_falloff/mask_od_mean/od_contrast, which
    # chromatin_od does not need. Pad sized to this window alone (25px), not od_ctx's 60px.
    t7a = time.perf_counter()
    hem_pad = cv2.copyMakeBorder(hem, OD_PAD_51, OD_PAD_51, OD_PAD_51, OD_PAD_51, cv2.BORDER_REPLICATE)
    px = pool['cx'].to_numpy() + OD_PAD_51
    py = pool['cy'].to_numpy() + OD_PAD_51
    stages['t7a_od_pad_s'] = time.perf_counter() - t7a

    t7b = time.perf_counter()
    pool['od51'] = [cm.chromatin_density(hem_pad, x, y, window=OD51_WINDOW) for x, y in zip(px, py)]
    stages['t7b_od51_loop_s'] = time.perf_counter() - t7b

    # ---- Stage 8: rank + top-100 by chromatin_od (od51) --------------------------------
    t8 = time.perf_counter()
    top_od51 = pool.sort_values('od51', ascending=False, na_position='last',
                                kind='mergesort').head(BUDGET)
    stages['t8_chromatin_od_rank_top100_s'] = time.perf_counter() - t8

    t_outer_combined_total = time.perf_counter() - t_outer0_tm
    gc.enable()
    cpu_limit_end = cpu_speed_limit()

    n_detections = len(pool)
    n_top_tm = len(top_tm)
    n_top_od51 = len(top_od51)
    tm_stage_keys = ['t1_refine_seed_box_s', 't2_patch_template_build_s', 't3_template_matching_s',
                     't4_threshold_peak_extraction_s', 't5_nms_selfhit_s', 't6_rank_top100_s']
    od_stage_keys = ['t7a_od_pad_s', 't7b_od51_loop_s', 't8_chromatin_od_rank_top100_s']
    t_tm_score_total = sum(stages[k] for k in tm_stage_keys)
    t_chromatin_od_overhead = sum(stages[k] for k in od_stage_keys)
    t_combined_total = t_tm_score_total + t_chromatin_od_overhead

    # Correctness: exact production metric (compare._tie_and_nan), order-independent.
    od51_tie, od51_nan_rate = cp._tie_and_nan(pool['od51'])

    del hem, gray_inv, hem_p, fused_p, valid_p, fused, valid, templates, patch, centers, scores, c, s, hem_pad
    gc.collect()

    meta = dict(
        file_name=fn, tumor_type=domain,
        n_detections=n_detections, base_size=base_size, n_retries=n_retries,
        n_top_tm=n_top_tm, n_top_od51=n_top_od51,
        seed_ann_id=seed_ann_id, map_median=round(float(med), 5), mad_scale=round(float(mad), 5),
        chromatin_od_nan_rate=od51_nan_rate, chromatin_od_largest_tie_block=od51_tie,
        cpu_limit_start=cpu_limit_start, cpu_limit_end=cpu_limit_end,
        t_setup_s=round(t_setup, 5),
        **{k: round(v, 5) for k, v in stages.items()},
        t_tm_score_total_s=round(t_tm_score_total, 5),
        t_chromatin_od_overhead_s=round(t_chromatin_od_overhead, 5),
        t_combined_total_s=round(t_combined_total, 5),
        t_outer_tm_total_s=round(t_outer_tm_total, 5),
        t_outer_combined_total_s=round(t_outer_combined_total, 5),
    )
    print(f"[{fn}] {domain:32s} n_det={n_detections:6d} "
          f"tm_total={t_tm_score_total*1000:7.1f}ms od_overhead={t_chromatin_od_overhead*1000:7.1f}ms "
          f"combined={t_combined_total*1000:7.1f}ms (+{100*t_chromatin_od_overhead/t_tm_score_total:.0f}%) "
          f"cpu_limit={cpu_limit_start}->{cpu_limit_end}",
          flush=True)
    return meta

## Run — all 14 ROIs

Same per-ROI cooldown gate as `od_contrast_latency_profile.ipynb`: `wait_for_cool_cpu()` before every ROI, since we should not assume this workload is light enough to skip the protection that the heavier `od_contrast` run needed.

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

rows = []
for fn in files:
    limit, waited = wait_for_cool_cpu()
    if waited > 0:
        print(f'  ...paused {waited}s before {fn} for CPU to cool (CPU_Speed_Limit now {limit})', flush=True)
    if limit != 100:
        print(f'  !! proceeding with {fn} despite CPU_Speed_Limit={limit} after {waited}s wait '
              f'-- the thermal-integrity gate below will catch and fail this run', flush=True)
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    rows.append(time_roi(fn, image_id, domain, annotations))
    gc.collect()

TIMING = pd.DataFrame(rows).set_index('file_name')
TIMING.to_csv('chromatin_od_latency_per_roi.csv')
print(f'\n{len(TIMING)} ROIs timed -> chromatin_od_latency_per_roi.csv')

[013.tiff] human breast cancer              n_det= 17724 tm_total= 1915.5ms od_overhead=  744.2ms combined= 2659.7ms (+39%) cpu_limit=100->100


[094.tiff] human breast cancer              n_det= 18628 tm_total= 1692.3ms od_overhead=  657.3ms combined= 2349.6ms (+39%) cpu_limit=100->100


[201.tiff] canine lung cancer               n_det= 15848 tm_total= 1513.4ms od_overhead=  561.1ms combined= 2074.5ms (+37%) cpu_limit=100->100


[233.tiff] canine lung cancer               n_det= 17449 tm_total= 1383.7ms od_overhead=  612.8ms combined= 1996.5ms (+44%) cpu_limit=100->100


[245.tiff] canine lymphosarcoma             n_det= 17678 tm_total= 1676.6ms od_overhead=  656.6ms combined= 2333.2ms (+39%) cpu_limit=100->100


...

[459.tiff] canine soft tissue sarcoma       n_det= 17806 tm_total= 1367.5ms od_overhead=  630.4ms combined= 1997.9ms (+46%) cpu_limit=100->100


[460.tiff] canine soft tissue sarcoma       n_det= 15698 tm_total= 1441.0ms od_overhead=  564.2ms combined= 2005.3ms (+39%) cpu_limit=100->100


[529.tiff] human melanoma                   n_det= 16620 tm_total= 1637.1ms od_overhead=  651.2ms combined= 2288.3ms (+40%) cpu_limit=100->100


[548.tiff] human melanoma                   n_det= 17839 tm_total= 1645.0ms od_overhead=  682.2ms combined= 2327.2ms (+41%) cpu_limit=100->100



14 ROIs timed -> chromatin_od_latency_per_roi.csv


## Thermal-integrity gate

Same hard gate as `od_contrast_latency_profile.ipynb`: `cpu_speed_limit()` was sampled
immediately before and after every ROI's timed work, outside all stage timers. If any ROI
ran at anything other than full speed, the run is discarded here.

In [5]:
cpu_ok = (TIMING['cpu_limit_start'] == 100) & (TIMING['cpu_limit_end'] == 100)
print(TIMING[['cpu_limit_start', 'cpu_limit_end']])
if not cpu_ok.all():
    print('\n!! THERMAL THROTTLING DETECTED -- these results are NOT trustworthy:')
    print(TIMING.loc[~cpu_ok, ['cpu_limit_start', 'cpu_limit_end']])
    raise AssertionError(
        'CPU was thermally throttled during at least one ROI (CPU_Speed_Limit < 100). '
        'Let the machine cool and re-execute this notebook from a fresh kernel.')
print(f'\nAll 14 ROIs ran at full CPU speed (CPU_Speed_Limit=100) throughout -- timings below are trustworthy.')

           cpu_limit_start  cpu_limit_end
file_name                                
013.tiff               100            100
094.tiff               100            100
201.tiff               100            100
233.tiff               100            100
245.tiff               100            100
246.tiff               100            100
300.tiff               100            100
301.tiff               100            100
402.tiff               100            100
403.tiff               100            100
459.tiff               100            100
460.tiff               100            100
529.tiff               100            100
548.tiff               100            100

All 14 ROIs ran at full CPU speed (CPU_Speed_Limit=100) throughout -- timings below are trustworthy.


## Cross-check 1 — stages 1-6 reproduce the accepted notebook exactly

In [6]:
oracle = pd.read_csv(ORACLE_RAW_CSV)
oracle_roi = (oracle.drop_duplicates('file_name')
              .set_index('file_name')[['seed_ann_id', 'base_size', 'n_detections',
                                        'map_median', 'mad_scale']])
cmp1 = TIMING[['seed_ann_id', 'base_size', 'n_detections', 'map_median', 'mad_scale']].join(
    oracle_roi, lsuffix='_this', rsuffix='_oracle')

mismatches = []
for col in ['seed_ann_id', 'base_size', 'n_detections']:
    bad = cmp1[f'{col}_this'].astype(int) != cmp1[f'{col}_oracle'].astype(int)
    if bad.any():
        mismatches.append((col, cmp1.index[bad].tolist()))
for col in ['map_median', 'mad_scale']:
    bad = ~np.isclose(cmp1[f'{col}_this'], cmp1[f'{col}_oracle'], rtol=0, atol=1e-5)
    if bad.any():
        mismatches.append((col, cmp1.index[bad].tolist()))

if mismatches:
    print('!! MISMATCH vs oracle (stages 1-6):')
    for col, rois in mismatches:
        print(f'   {col}: {rois}')
    display(cmp1)
    raise AssertionError('stages 1-6 do not reproduce the accepted pipeline')

print(f'All {len(cmp1)} ROIs match {ORACLE_RAW_CSV} exactly on '
      f'seed_ann_id/base_size/n_detections/map_median/mad_scale.')

All 14 ROIs match ../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv exactly on seed_ann_id/base_size/n_detections/map_median/mad_scale.


## Cross-check 2 — chromatin_od (od51) is computed correctly

Same method as the `od_contrast` notebook's cross-check 2: `nan_rate`/`largest_tie_block`
via `compare._tie_and_nan`, compared against production's committed `arm='chromatin_od'`
row. These are exact properties of the `od51` values themselves, sensitive to every
candidate's floating-point value -- a match is a strong correctness signal, not just a
shape check. Also verifies the minimal 25px pad didn't silently drop coverage anywhere
(`nan_rate` must be exactly 0, as the smoke test found).

In [7]:
oracle_od = (oracle[oracle['arm'] == 'chromatin_od']
             .drop_duplicates('file_name')
             .set_index('file_name')[['nan_rate', 'largest_tie_block']])
cmp2 = TIMING[['chromatin_od_nan_rate', 'chromatin_od_largest_tie_block']].join(oracle_od)

nan_bad = ~np.isclose(cmp2['chromatin_od_nan_rate'], cmp2['nan_rate'], atol=1e-9)
tie_bad = cmp2['chromatin_od_largest_tie_block'].astype(int) != cmp2['largest_tie_block'].astype(int)

if nan_bad.any() or tie_bad.any():
    print('!! MISMATCH vs oracle (chromatin_od) -- stage 7 diverged from production:')
    display(cmp2[nan_bad | tie_bad])
    raise AssertionError('chromatin_od does not reproduce the accepted pipeline')

print(f'All {len(cmp2)} ROIs match the oracle exactly on chromatin_od nan_rate and '
      f'largest_tie_block -- the minimal-pad stage 7 reproduces production.')
cmp2

All 14 ROIs match the oracle exactly on chromatin_od nan_rate and largest_tie_block -- the minimal-pad stage 7 reproduces production.


,chromatin_od_nan_rate,chromatin_od_largest_tie_block,nan_rate,largest_tie_block
file_name,,,,
013.tiff,0.0,2,0.0,2
094.tiff,0.0,2,0.0,2
201.tiff,0.0,2,0.0,2
233.tiff,0.0,2,0.0,2
245.tiff,0.0,2,0.0,2
246.tiff,0.0,2,0.0,2
300.tiff,0.0,2,0.0,2
301.tiff,0.0,2,0.0,2
402.tiff,0.0,2,0.0,2


## Cross-check 3 — verifying the prior-turn estimate against what actually ran

Before this notebook existed, a same-conversation estimate reused `od_contrast`'s own
measured `t7a+t7b+t8` (built with a 60px pad sized for `od_ctx`) as a stand-in for
`chromatin_od`'s cost, arriving at ~720ms mean. That reuse was flagged explicitly as an
estimate, not a measurement. This compares it, per ROI, against what actually ran here.

In [8]:
od_contrast_per_roi = pd.read_csv(OD_CONTRAST_PER_ROI_CSV).set_index('file_name')
# od_contrast_latency_per_roi.csv was saved from the seconds-based TIMING frame (not the
# ms-based TIMING_MS display frame), so its columns carry the _s suffix -- verified against
# the file's actual header before writing this, not assumed.
assert 't7a_od_pad_s' in od_contrast_per_roi.columns, (
    f'expected t7a_od_pad_s in {OD_CONTRAST_PER_ROI_CSV}, found: {list(od_contrast_per_roi.columns)}')
estimate_ms = (od_contrast_per_roi['t7a_od_pad_s'] + od_contrast_per_roi['t7b_od51_loop_s']
              + od_contrast_per_roi['t8_od_contrast_rank_top100_s']) * 1000
actual_ms = TIMING['t_chromatin_od_overhead_s'] * 1000

est_vs_actual = pd.DataFrame({
    'estimate_ms (reused od_contrast t7a/t7b/t8, 60px pad)': estimate_ms.round(1),
    'actual_ms (this notebook, 25px pad)': actual_ms.round(1),
    'delta_ms': (actual_ms - estimate_ms).round(1),
    'delta_pct': ((actual_ms - estimate_ms) / estimate_ms * 100).round(1),
})
print(f'Mean estimate: {estimate_ms.mean():.0f}ms | Mean actual: {actual_ms.mean():.0f}ms | '
      f'Mean delta: {(actual_ms - estimate_ms).mean():.0f}ms '
      f'({(actual_ms - estimate_ms).mean() / estimate_ms.mean() * 100:.0f}%)')
est_vs_actual

Mean estimate: 721ms | Mean actual: 645ms | Mean delta: -76ms (-11%)


,"estimate_ms (reused od_contrast t7a/t7b/t8, 60px pad)","actual_ms (this notebook, 25px pad)",delta_ms,delta_pct
file_name,,,,
013.tiff,960.1,744.2,-216.0,-22.5
094.tiff,791.8,657.3,-134.5,-17.0
201.tiff,623.1,561.1,-62.0,-9.9
233.tiff,654.4,612.8,-41.6,-6.4
245.tiff,666.6,656.6,-10.0,-1.5
246.tiff,971.8,653.0,-318.8,-32.8
300.tiff,678.9,643.9,-35.0,-5.1
301.tiff,663.7,630.1,-33.5,-5.1
402.tiff,685.6,683.7,-1.9,-0.3


## Table 1 — per-ROI, per-stage timing (ms)

In [9]:
TM_STAGE_COLS_S = ['t1_refine_seed_box_s', 't2_patch_template_build_s', 't3_template_matching_s',
                   't4_threshold_peak_extraction_s', 't5_nms_selfhit_s', 't6_rank_top100_s']
OD_STAGE_COLS_S = ['t7a_od_pad_s', 't7b_od51_loop_s', 't8_chromatin_od_rank_top100_s']
TOTAL_COLS_S = ['t_tm_score_total_s', 't_chromatin_od_overhead_s', 't_combined_total_s',
                't_outer_tm_total_s', 't_outer_combined_total_s']
ALL_TIME_COLS_S = ['t_setup_s'] + TM_STAGE_COLS_S + OD_STAGE_COLS_S + TOTAL_COLS_S

TIMING_MS = TIMING.copy()
for c in ALL_TIME_COLS_S:
    TIMING_MS[c[:-2] + '_ms'] = (TIMING_MS[c] * 1000).round(1)
TM_STAGE_COLS_MS = [c[:-2] + '_ms' for c in TM_STAGE_COLS_S]
OD_STAGE_COLS_MS = [c[:-2] + '_ms' for c in OD_STAGE_COLS_S]
ALL_TIME_COLS_MS = [c[:-2] + '_ms' for c in ALL_TIME_COLS_S]

TIMING_MS['pct_overhead'] = (TIMING_MS['t_chromatin_od_overhead_ms']
                             / TIMING_MS['t_tm_score_total_ms'] * 100).round(1)

display_cols = (['tumor_type', 'n_detections'] + TM_STAGE_COLS_MS + ['t_tm_score_total_ms']
                + OD_STAGE_COLS_MS + ['t_chromatin_od_overhead_ms', 'pct_overhead', 't_combined_total_ms'])
TIMING_MS[display_cols]

,tumor_type,n_detections,t1_refine_seed_box_ms,t2_patch_template_build_ms,t3_template_matching_ms,t4_threshold_peak_extraction_ms,t5_nms_selfhit_ms,t6_rank_top100_ms,t_tm_score_total_ms,t7a_od_pad_ms,t7b_od51_loop_ms,t8_chromatin_od_rank_top100_ms,t_chromatin_od_overhead_ms,pct_overhead,t_combined_total_ms
file_name,,,,,,,,,,,,,,,
013.tiff,human breast cancer,17724,1.0,0.0,1347.6,299.6,265.9,1.3,1915.6,95.4,646.6,2.2,744.2,38.8,2659.7
094.tiff,human breast cancer,18628,0.9,0.0,1085.7,284.2,320.5,1.0,1692.3,29.5,625.9,1.9,657.3,38.8,2349.6
201.tiff,canine lung cancer,15848,1.2,0.0,1144.7,235.6,130.8,1.1,1513.4,23.4,536.1,1.6,561.1,37.1,2074.5
233.tiff,canine lung cancer,17449,1.1,0.0,931.1,239.7,210.9,0.9,1383.7,24.4,586.7,1.8,612.8,44.3,1996.5
245.tiff,canine lymphosarcoma,17678,1.2,0.0,1142.9,247.8,283.6,1.0,1676.6,24.2,630.6,1.9,656.6,39.2,2333.2
246.tiff,canine lymphosarcoma,18013,1.6,0.0,1077.8,462.9,189.1,0.9,1732.4,24.8,626.2,2.0,653.0,37.7,2385.5
300.tiff,canine cutaneous mast cell tumor,17940,2.3,0.0,1031.8,226.2,140.2,0.9,1401.6,24.7,617.0,2.3,643.9,45.9,2045.5
301.tiff,canine cutaneous mast cell tumor,17710,1.3,0.0,997.6,228.3,153.9,1.0,1382.1,22.7,605.2,2.3,630.1,45.6,2012.2
402.tiff,human neuroendocrine tumor,17532,0.9,0.0,1125.9,279.1,235.6,1.1,1642.8,76.7,605.1,2.0,683.7,41.6,2326.4


## Table 2 — summary across all 14 ROIs (ms)

In [10]:
SUMMARY = TIMING_MS[ALL_TIME_COLS_MS + ['pct_overhead']].agg(['mean', 'std', 'min', 'max']).T
SUMMARY.to_csv('chromatin_od_latency_summary.csv')
SUMMARY.round(2)

,mean,std,min,max
t_setup_ms,3087.16,547.02,2601.9,4098.6
t1_refine_seed_box_ms,1.26,0.38,0.9,2.3
t2_patch_template_build_ms,0.00,0.00,0.0,0.0
t3_template_matching_ms,1121.84,128.46,931.1,1389.2
t4_threshold_peak_extraction_ms,272.02,60.32,226.2,462.9
t5_nms_selfhit_ms,192.32,64.38,104.2,320.5
t6_rank_top100_ms,1.01,0.15,0.8,1.3
t7a_od_pad_ms,45.46,29.73,22.7,95.4
t7b_od51_loop_ms,597.55,33.57,536.1,646.6
t8_chromatin_od_rank_top100_ms,1.96,0.23,1.6,2.3


## Table 3 — summary by tumor domain (ms)

In [11]:
BY_DOMAIN = TIMING_MS.groupby('tumor_type')[
    ['t_tm_score_total_ms', 't_chromatin_od_overhead_ms', 't_combined_total_ms', 'pct_overhead']
].agg(['mean', 'std', 'min', 'max'])
BY_DOMAIN.to_csv('chromatin_od_latency_by_domain.csv')
BY_DOMAIN.round(2)

t_tm_score_total_ms                         t_chromatin_od_overhead_ms                      t_combined_total_ms                         pct_overhead                  
                                                mean     std     min     max                       mean    std    min    max                mean     std     min     max         mean   std   min   max
tumor_type                                                                                                                                                                                             
canine cutaneous mast cell tumor             1391.85   13.79  1382.1  1401.6                     637.00   9.76  630.1  643.9             2028.85   23.55  2012.2  2045.5        45.75  0.21  45.6  45.9
canine lung cancer                           1448.55   91.71  1383.7  1513.4                     586.95  36.56  561.1  612.8             2035.50   55.15  1996.5  2074.5        40.70  5.09  37.1  44.3
canine lymphosarcoma                         1704.50   39.46  1676.6  1732.4                     654.80   2.55  653.0  656.6             2359.35   36.98  2333.2  2385.5        38.45  1.06  37.7  39.2
canine soft tissue sarcoma                   1404.30   51.90  1367.6  1441.0                     597.30  46.81  564.2  630.4             2001.60    5.23  1997.9  2005.3        42.65  4.88  39.2  46.1
human breast cancer                          1803.95  157.90  1692.3  1915.6                     700.75  61.45  657.3  744.2             2504.65  219.27  2349.6  2659.7        38.80  0.00  38.8  38.8
human melanoma                               1641.05    5.59  1637.1  1645.0                     666.70  21.92  651.2  682.2             2307.75   27.51  2288.3  2327.2        40.65  1.20  39.8  41.5
human neuroendocrine tumor                   1725.40  116.81  1642.8  1808.0                     671.05  17.89  658.4  683.7             2396.45   99.07  2326.4  2466.5        39.00  3.68  36.4  41.6

## Diagnostic checks

In [12]:
outer_vs_sum_tm = (TIMING_MS['t_outer_tm_total_ms'] - TIMING_MS['t_tm_score_total_ms']).abs()
outer_vs_sum_combined = (TIMING_MS['t_outer_combined_total_ms'] - TIMING_MS['t_combined_total_ms']).abs()
print(f'Outer-timer vs stage-sum cross-check (stages 1-6): max discrepancy '
      f'{outer_vs_sum_tm.max():.2f}ms across all 14 ROIs.')
print(f'Outer-timer vs stage-sum cross-check (stages 1-8): max discrepancy '
      f'{outer_vs_sum_combined.max():.2f}ms across all 14 ROIs.')

starved_tm = TIMING[TIMING['n_top_tm'] < BUDGET]
starved_od = TIMING[TIMING['n_top_od51'] < BUDGET]
print(f'\ntm_score top-{BUDGET} starved on {len(starved_tm)}/14 ROIs; '
      f'chromatin_od top-{BUDGET} starved on {len(starved_od)}/14 ROIs.')

print(f'\nSETUP included a retry search on {int((TIMING["n_retries"] > 0).sum())}/14 ROIs '
      f'(n_retries value counts: {TIMING["n_retries"].value_counts().to_dict()}).')

print('\nStage 7 breakdown, mean share of stage-7 time:')
s7_cols = ['t7a_od_pad_ms', 't7b_od51_loop_ms']
s7_mean = TIMING_MS[s7_cols].mean()
for c, v in (s7_mean / s7_mean.sum() * 100).items():
    print(f'  {c}: {v:.1f}%  (mean {s7_mean[c]:.1f}ms)')

Outer-timer vs stage-sum cross-check (stages 1-6): max discrepancy 0.10ms across all 14 ROIs.
Outer-timer vs stage-sum cross-check (stages 1-8): max discrepancy 0.10ms across all 14 ROIs.

tm_score top-100 starved on 0/14 ROIs; chromatin_od top-100 starved on 0/14 ROIs.

SETUP included a retry search on 1/14 ROIs (n_retries value counts: {0: 13, 1: 1}).

Stage 7 breakdown, mean share of stage-7 time:
  t7a_od_pad_ms: 7.1%  (mean 45.5ms)
  t7b_od51_loop_ms: 92.9%  (mean 597.6ms)


## Written readout

In [13]:
tm_mean = TIMING_MS['t_tm_score_total_ms'].mean()
od_overhead_mean = TIMING_MS['t_chromatin_od_overhead_ms'].mean()
od_overhead_min = TIMING_MS['t_chromatin_od_overhead_ms'].min()
od_overhead_max = TIMING_MS['t_chromatin_od_overhead_ms'].max()
combined_mean = TIMING_MS['t_combined_total_ms'].mean()
pct_mean = TIMING_MS['pct_overhead'].mean()
pct_min = TIMING_MS['pct_overhead'].min()
pct_max = TIMING_MS['pct_overhead'].max()
s3_mean = TIMING_MS['t3_template_matching_ms'].mean()

# Load od_contrast's own summary for a direct side-by-side (same 14 ROIs, same machine).
od_contrast_summary = pd.read_csv('od_contrast_latency_summary.csv', index_col=0)
od_contrast_overhead_mean = od_contrast_summary.loc['t_od_contrast_overhead_ms', 'mean']

print(f'chromatin_od re-ranking costs {od_overhead_mean:.0f}ms on average per ROI '
      f'(range {od_overhead_min:.0f}-{od_overhead_max:.0f}ms) on top of the {tm_mean:.0f}ms '
      f'tm_score-only pipeline -- a {pct_mean:.0f}% increase (range {pct_min:.0f}%-{pct_max:.0f}%), '
      f'taking the combined total to {combined_mean:.0f}ms.')
print()
print(f'Compared directly against od_contrast (same 14 ROIs, same machine, same session): '
      f'the chromatin_od overhead ({od_overhead_mean:.0f}ms) is '
      f'{od_contrast_overhead_mean / od_overhead_mean:.1f}x cheaper than the od_contrast overhead '
      f'({od_contrast_overhead_mean:.0f}ms) -- confirms the mechanism: chromatin_od skips the '
      f'od_ctx loop (121x121 window) entirely, computing only od51 (51x51 window).')
print()
if od_overhead_mean > s3_mean:
    verdict = f'exceeds it ({od_overhead_mean:.0f}ms vs {s3_mean:.0f}ms)'
elif od_overhead_mean > 0.5 * s3_mean:
    verdict = f'is a large fraction of it ({od_overhead_mean:.0f}ms vs {s3_mean:.0f}ms) without overtaking it'
else:
    verdict = f'stays comfortably below it ({od_overhead_mean:.0f}ms vs {s3_mean:.0f}ms)'
print(f'Against template matching (stage 3, {s3_mean:.0f}ms mean, still the dominant stage in the '
      f'tm_score-only pipeline): the chromatin_od overhead {verdict}. Unlike od_contrast, '
      f'chromatin_od does not become the new dominant stage.')
print()
print(f'The prior-turn estimate (reusing t7a/t7b/t8 from the od_contrast run, 60px pad) is checked '
      f'in Cross-check 3 above against what this notebook actually measured with the correct, '
      f'minimal 25px pad -- see that cell for the per-ROI delta.')

chromatin_od re-ranking costs 645ms on average per ROI (range 561-744ms) on top of the 1589ms tm_score-only pipeline -- a 41% increase (range 36%-46%), taking the combined total to 2233ms.



Compared directly against od_contrast (same 14 ROIs, same machine, same session): the chromatin_od overhead (645ms) is 5.4x cheaper than the od_contrast overhead (3462ms) -- confirms the mechanism: chromatin_od skips the od_ctx loop (121x121 window) entirely, computing only od51 (51x51 window).

Against template matching (stage 3, 1122ms mean, still the dominant stage in the tm_score-only pipeline): the chromatin_od overhead is a large fraction of it (645ms vs 1122ms) without overtaking it. Unlike od_contrast, chromatin_od does not become the new dominant stage.

The prior-turn estimate (reusing t7a/t7b/t8 from the od_contrast run, 60px pad) is checked in Cross-check 3 above against what this notebook actually measured with the correct, minimal 25px pad -- see that cell for the per-ROI delta.
